# TurnWave Phase 4 — retrain at the larger budget

**Runtime -> Change runtime type -> T4 GPU** before running anything.

Phase 2's acoustic branch peaked on its *final* step with train and validation
loss still together: it was under-trained, not overfit. This run doubles the
data (60k -> 120k) and the steps (4k -> 8k), and pays for it by cutting the text
branch to 1,500 steps -- its AP peaked at step 1,250, so the rest was waste.

Cuts now land 0.2 s into each pause, matching eot-bench and real endpointing.

~2h05m total. `best.pt` is written every 250 steps, so the run can be stopped
at any point and the best checkpoint is already on disk.

In [ ]:
# 1. Setup (~3 min). Fails loudly rather than three cells later.
import os, subprocess, sys

if not os.path.isdir('/content/turnwave'):
    !git clone -q https://github.com/Nikhils-G/turnwave.git /content/turnwave
%cd /content/turnwave
!git pull -q

install = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.',
                          '--no-deps', 'sentencepiece', 'datasets', 'soundfile',
                          'onnx', 'onnxruntime', 'onnxscript'])
assert install.returncode == 0, 'install failed - read the error above'

import torch
assert torch.cuda.is_available(), 'No GPU. Runtime -> Change runtime type -> T4 GPU.'
print('GPU OK:', torch.cuda.get_device_name(0))

In [ ]:
# 2. Text branch (~14 min). 1,500 steps: the 3,500-step run peaked at 1,250.
!python scripts/build_text_dataset.py --out data/text && \
 python -m turnwave.tokenizer data/text/corpus.txt checkpoints/tokenizer && \
 python -m turnwave.train --task text \
     --train data/text/train.jsonl --val data/text/validation.jsonl \
     --tokenizer checkpoints/tokenizer/spm.model --out checkpoints/text_eot \
     --steps 1500 --batch-size 256 --num-workers 2

In [ ]:
# 3. Feature cache, doubled (~10 min). --cut-offset 0.2 is the default; passed
# explicitly so the manifest and this notebook agree about what was built.
!python scripts/build_audio_dataset.py --out data/audio --cut-offset 0.2 \
    --max-examples 120000 --max-eval-examples 8000

In [ ]:
# 4. Acoustic branch, doubled steps (~70 min). Gate: val AP must beat 0.741.
!python -m turnwave.train --task audio --cache data/audio \
    --out checkpoints/audio_eot \
    --steps 8000 --batch-size 128 --lr 3e-4 --num-workers 2

In [ ]:
# 5. Fusion head (~25 min).
!python -m turnwave.train --task fusion --cache data/audio \
    --tokenizer checkpoints/tokenizer/spm.model \
    --text-ckpt checkpoints/text_eot/best.pt \
    --audio-ckpt checkpoints/audio_eot/best.pt \
    --out checkpoints/fusion_eot \
    --steps 2500 --batch-size 128 --lr 1e-3 --num-workers 2

In [ ]:
# 6. The ablation, then ONNX with measured CPU latency.
!python -m turnwave.ablate --cache data/audio \
    --tokenizer checkpoints/tokenizer/spm.model \
    --text-ckpt checkpoints/text_eot/best.pt \
    --audio-ckpt checkpoints/audio_eot/best.pt \
    --fusion-ckpt checkpoints/fusion_eot/best.pt \
    --split test --device cuda --out docs/ablation.json
!python -m turnwave.export --ckpt checkpoints/text_eot/best.pt   --out-dir checkpoints/onnx
!python -m turnwave.export --ckpt checkpoints/audio_eot/best.pt  --out-dir checkpoints/onnx
!python -m turnwave.export --ckpt checkpoints/fusion_eot/best.pt --out-dir checkpoints/onnx

In [ ]:
# 7. Curves, then save everything. Titles are derived from each run's own data.
!python scripts/plot_training.py checkpoints/text_eot/log.csv   --out docs/training_curves.png
!python scripts/plot_training.py checkpoints/audio_eot/log.csv  --out docs/audio_curves.png
!python scripts/plot_training.py checkpoints/fusion_eot/log.csv --out docs/fusion_curves.png
!zip -qr turnwave_phase4.zip checkpoints docs
from google.colab import files
files.download('turnwave_phase4.zip')